# Time after transplantation

## Setup

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.stats.multitest import multipletests

ROOT = next(
    folder for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (folder / "result_summary" / "data_helper.py").exists()
)
sys.path[:0] = [
    str(ROOT / "result_summary"),
    str(ROOT / "result_summary" / "differential_rules"),
]

import data_helper as dh
import differential_stats as ds
import differential_vis as dv

warnings.filterwarnings("ignore")

ORGANS = ["Colon", "Duodenum"]
TIME_GROUPS = ["<30", "30-100", ">100"]
DISPLAY_GROUPS = ["Control", *TIME_GROUPS]
MIN_CELLS = 20
MIN_BIOPSIES = 3
MIN_RULE_BIOPSIES = 3
N_PERMUTATIONS = 5_000
FDR_CUTOFF = 0.05
TOP_RULES = 6

TEX_TIME_EXPORTS = {
    ("Colon", "CD8T -> Goblet"): "tex_new_time_colon_cd8_goblet.pdf",
    ("Colon", "SMV -> Endothelial"): "tex_new_time_colon_smv_endothelial.pdf",
    ("Colon", "SMV -> Goblet"): "tex_new_time_colon_smv_goblet.pdf",
}

POOLED_CONTRASTS = {
    "before_100_vs_after": {
        "title": "<30 + 30–100 days  vs  >100 days",
        "a_name": "≤100 days", "b_name": ">100 days",
        "a_groups": ["<30", "30-100"], "b_groups": [">100"],
    },
    "before_30_vs_after": {
        "title": "<30 days  vs  30–100 + >100 days",
        "a_name": "<30 days", "b_name": "≥30 days",
        "a_groups": ["<30"], "b_groups": ["30-100", ">100"],
    },
}

## Data

In [ ]:
df_cells, df_fovs, df_biopsy = dh.load_spatial_data()
df_results = dh.load_results(rule_max_items=2, kind=None)
rules = dh.prepare_rules(df_results)
_, eligibility, eligible_states = ds.state_tables(
    rules, df_cells, df_fovs["FOV"], MIN_CELLS
)

days = df_biopsy.set_index("Biopsy_ID")["Days after Transplant"]
df_fovs["Days after Transplant"] = df_fovs["Biopsy"].map(days)
biopsy_metadata = (
    df_fovs[["Biopsy", "Organ", "Days after Transplant", "Days after Transplant grouped"]]
    .drop_duplicates(["Biopsy", "Organ"])
    .rename(columns={"Days after Transplant grouped": "Time group"})
)

In [ ]:
def by_biopsy(states, organ):
    """Average eligible FOV states so each biopsy contributes one value."""
    metadata = df_fovs[df_fovs["Organ"] == organ]
    return pd.DataFrame({
        biopsy: states[rows["FOV"]].mean(axis=1)
        for biopsy, rows in metadata.groupby("Biopsy")
    })

In [ ]:
def permutation_pvalues(values, labels, observed, seed=0):
    """Largest difference between the three time-window means."""
    rng = np.random.default_rng(seed)
    extreme = np.zeros(len(observed), int)
    valid_count = np.zeros(len(observed), int)
    for _ in range(N_PERMUTATIONS):
        shuffled = rng.permutation(labels)
        means = np.column_stack([
            np.nanmean(values[:, shuffled == group], axis=1)
            for group in TIME_GROUPS
        ])
        null = np.nanmax(means, axis=1) - np.nanmin(means, axis=1)
        valid = np.isfinite(null)
        valid_count += valid
        extreme += valid & (null >= observed - 1e-12)
    return (extreme + 1) / (valid_count + 1)

In [ ]:
def analyze_time(organ, values):
    """Biopsy-level temporal screen with one FDR correction across all rules."""
    metadata = biopsy_metadata[
        (biopsy_metadata["Organ"] == organ)
        & biopsy_metadata["Time group"].isin(TIME_GROUPS)
    ]
    biopsies = metadata["Biopsy"].tolist()
    labels = metadata["Time group"].to_numpy()
    matrix = values.reindex(columns=biopsies).to_numpy(float)

    eligible_n = {
        group: np.isfinite(matrix[:, labels == group]).sum(axis=1)
        for group in TIME_GROUPS
    }
    keep = np.sum(np.isfinite(matrix) & (matrix != 0), axis=1) >= MIN_RULE_BIOPSIES
    for count in eligible_n.values():
        keep &= count >= MIN_BIOPSIES

    matrix = matrix[keep]
    names = values.index[keep]
    means = np.column_stack([
        np.nanmean(matrix[:, labels == group], axis=1)
        for group in TIME_GROUPS
    ])
    gap = np.nanmax(means, axis=1) - np.nanmin(means, axis=1)
    p_value = permutation_pvalues(matrix, labels, gap)

    result = pd.DataFrame(index=names)
    for i, group in enumerate(TIME_GROUPS):
        result[f"net_{group}"] = means[:, i]
        result[f"n_eligible_{group}"] = eligible_n[group][keep]
    result["range"] = gap
    result["lowest"] = np.array(TIME_GROUPS)[np.nanargmin(means, axis=1)]
    result["highest"] = np.array(TIME_GROUPS)[np.nanargmax(means, axis=1)]
    result["p_value"] = p_value
    result["fdr"] = multipletests(p_value, method="fdr_bh")[1]
    return result.sort_values(["fdr", "range"], ascending=[True, False])

In [ ]:
def strongest(result):
    passed = result[result["fdr"] <= FDR_CUTOFF]
    return (passed if len(passed) else result).head(TOP_RULES).index.tolist()

In [ ]:
def result_table(result, rows=12):
    table = result[[
        "net_<30", "net_30-100", "net_>100", "range",
        "lowest", "highest", "fdr",
    ]].head(rows).copy()
    for column in ["net_<30", "net_30-100", "net_>100", "range"]:
        table[column] = (100 * table[column]).round(1)
    return table.rename(columns={
        "net_<30": "<30 net %", "net_30-100": "30–100 net %",
        "net_>100": ">100 net %", "range": "largest gap (pp)",
        "fdr": "temporal FDR",
    })

In [ ]:
def analyze_contrast(organ, values, spec):
    metadata = biopsy_metadata[
        (biopsy_metadata["Organ"] == organ)
        & biopsy_metadata["Time group"].isin(TIME_GROUPS)
    ]
    a = metadata.loc[metadata["Time group"].isin(spec["a_groups"]), "Biopsy"].tolist()
    b = metadata.loc[metadata["Time group"].isin(spec["b_groups"]), "Biopsy"].tolist()
    return ds.compare(
        values, a, b, "a", "b",
        min_eligible=MIN_BIOPSIES,
        min_present=MIN_RULE_BIOPSIES,
        n_permutations=N_PERMUTATIONS,
    )

In [ ]:
def analyze_pooled_contrasts(organ, values):
    """Apply one FDR correction across both pooled contrasts and all their rules."""
    results = {
        name: analyze_contrast(organ, values, spec)
        for name, spec in POOLED_CONTRASTS.items()
    }
    all_p = pd.concat(
        {name: result["p_value"] for name, result in results.items()},
        names=["contrast", "rule"],
    )
    corrected = pd.Series(
        multipletests(all_p, method="fdr_bh")[1], index=all_p.index
    )
    for name, result in results.items():
        result["fdr"] = corrected.xs(name).reindex(result.index)
        results[name] = result.sort_values(
            ["fdr", "effect_size"],
            key=lambda s: s.abs() if s.name == "effect_size" else s,
            ascending=[True, False],
        )
    return results

In [ ]:
def pooled_table(results, rows=8):
    """Compact result table for both pooled contrasts."""
    tables = []
    for name, result in results.items():
        table = result.head(rows).copy()
        table.insert(0, "contrast", POOLED_CONTRASTS[name]["title"])
        table.insert(1, "rule", table.index)
        tables.append(table)
    table = pd.concat(tables, ignore_index=True)
    for column in ["net_a", "net_b", "effect_size"]:
        table[column] = (100 * table[column]).round(1)
    return table[[
        "contrast", "rule", "net_a", "net_b", "effect_size",
        "n_eligible_a", "n_eligible_b", "fdr",
    ]].rename(columns={
        "net_a": "first group net %",
        "net_b": "second group net %",
        "effect_size": "gap (pp)",
        "fdr": "joint FDR",
    })

In [ ]:
biopsy_values = {organ: by_biopsy(eligible_states, organ) for organ in ORGANS}
time_results = {organ: analyze_time(organ, biopsy_values[organ]) for organ in ORGANS}
pooled_results = {
    organ: analyze_pooled_contrasts(organ, biopsy_values[organ])
    for organ in ORGANS
}

display(
    biopsy_metadata.groupby(["Organ", "Time group"], observed=False)
    .size().unstack(fill_value=0).reindex(columns=DISPLAY_GROUPS, fill_value=0)
)
for organ in ORGANS:
    passed = int((time_results[organ]["fdr"] <= FDR_CUTOFF).sum())
    pooled_passed = sum(
        int((result["fdr"] <= FDR_CUTOFF).sum())
        for result in pooled_results[organ].values()
    )
    print(f"{organ}: {passed} temporal and {pooled_passed} pooled results pass FDR {FDR_CUTOFF}")

## Biopsy coverage

In [ ]:
dv.plot_time_coverage(
    biopsy_metadata,
    day_col="Days after Transplant",
    group_col="Time group",
    groups=TIME_GROUPS,
)


## Full temporal screen

### Colon

In [ ]:
display(result_table(time_results["Colon"]))

dv.plot_temporal_screen(
    time_results["Colon"], scope="Colon", fdr_threshold=FDR_CUTOFF,
)

colon_metadata = biopsy_metadata[biopsy_metadata["Organ"] == "Colon"]
for rule in strongest(time_results["Colon"]):
    dv.plot_time_profiles(
        biopsy_values["Colon"], colon_metadata, [rule],
        time_results["Colon"], DISPLAY_GROUPS, group_col="Time group",
        scope="Colon — largest exploratory changes",
        save=TEX_TIME_EXPORTS.get(("Colon", rule)),
    )


### Duodenum

In [ ]:
display(result_table(time_results["Duodenum"]))

dv.plot_temporal_screen(
    time_results["Duodenum"], scope="Duodenum", fdr_threshold=FDR_CUTOFF,
)

duodenum_metadata = biopsy_metadata[biopsy_metadata["Organ"] == "Duodenum"]
for rule in strongest(time_results["Duodenum"]):
    dv.plot_time_profiles(
        biopsy_values["Duodenum"], duodenum_metadata, [rule],
        time_results["Duodenum"], DISPLAY_GROUPS, group_col="Time group",
        scope="Duodenum — largest exploratory changes",
    )


## Pooled time windows

### Colon

In [ ]:
display(pooled_table(pooled_results["Colon"]))

dv.plot_pooled_time_contrasts(
    pooled_results["Colon"], POOLED_CONTRASTS, organ="Colon",
    fdr_threshold=FDR_CUTOFF,
)


### Duodenum

In [ ]:
display(pooled_table(pooled_results["Duodenum"]))

dv.plot_pooled_time_contrasts(
    pooled_results["Duodenum"], POOLED_CONTRASTS, organ="Duodenum",
    fdr_threshold=FDR_CUTOFF, save="tex_new_time_duodenum_pooled.pdf",
)


## Paper-linked immune rules

In [ ]:
requested = {
    "Colon": [
        "CD4T -> Plasma", "CD8T -> Plasma",
        "Macrophage -> CD4T", "CD8T -> Macrophage",
    ],
    "Duodenum": [
        "CD4T -> Plasma", "Macrophage -> Plasma",
        "Macrophage -> CD4T", "CD8T -> Epithelial",
    ],
}

for number, organ in enumerate(ORGANS, start=8):
    metadata = biopsy_metadata[biopsy_metadata["Organ"] == organ]
    available = [rule for rule in requested[organ] if rule in biopsy_values[organ].index]
    dv.plot_time_profiles(
        biopsy_values[organ], metadata, available, time_results[organ],
        DISPLAY_GROUPS, group_col="Time group",
        scope=f"{organ} — paper-motivated immune context",
    )
